# 02 - Preprocessing & Baseline

Este notebook implementa:
1. Carga y exploracion del dataset
2. Feature engineering (date features, investor count, valuation cleaning)
3. Definicion de X e y (target: valuation_b)
4. Pipeline de preprocesamiento (imputacion, escalado, OneHotEncoder)
5. Entrenamiento de baselines: DummyRegressor, LinearRegression, Ridge
6. Evaluacion con MAE, MSE, RMSE, R2
7. Comparacion train/test para detectar overfitting
8. Guardado del modelo con joblib

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

# Ajustar path para imports desde la raiz del proyecto
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing.preprocessing_pipeline import (
    FeatureEngineer,
    get_feature_types,
    build_full_pipeline,
)
from src.models.evaluate import compare_train_test, detect_overfitting, print_evaluation_report

print('Librerias cargadas correctamente.')

## 1. Carga del Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download('ramjasmaurya/unicorn-startups')
csv_file = os.path.join(path, 'unicorns till sep 2022.csv')
df = pd.read_csv(csv_file)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Columnas:', df.columns.tolist())
print()
print(df.info())
print()
print('Valores nulos:')
print(df.isnull().sum())

## 2. Feature Engineering

In [ ]:
fe = FeatureEngineer(reference_date='2022-09-01')
df_eng = fe.fit_transform(df)

print(f'Despues de feature engineering: {df_eng.shape}')
print(f'Columnas: {df_eng.columns.tolist()}')
df_eng.head()

In [ ]:
print('Tipos de datos:')
print(df_eng.dtypes)
print()
print('Estadisticas descriptivas:')
df_eng.describe()

## 3. Definicion de X e y

In [ ]:
y = df_eng['valuation_b']
X = df_eng.drop(columns=['valuation_b'])

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Features: {X.columns.tolist()}')
print(f'Target: valuation_b (min={y.min():.2f}, max={y.max():.2f}, mean={y.mean():.2f})')

## 4. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape[0]} muestras')
print(f'Test:  {X_test.shape[0]} muestras')

## 5. Pipeline de Preprocesamiento

In [ ]:
numeric_cols, categorical_cols = get_feature_types(X)
print(f'Numericas: {numeric_cols}')
print(f'Categoricas: {categorical_cols}')

pipeline = build_full_pipeline(numeric_cols, categorical_cols)
pipeline.fit(X_train, y_train)

X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

print(f'\nX_train transformado: {X_train_transformed.shape}')
print(f'X_test transformado:  {X_test_transformed.shape}')
print(f'Valores nulos en train: {np.isnan(X_train_transformed).sum()}')
print(f'Valores nulos en test:  {np.isnan(X_test_transformed).sum()}')

## 6. Entrenamiento de Baselines

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    'DummyRegressor': DummyRegressor(strategy='mean'),
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
}

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', pipeline),
        ('model', model),
    ])
    pipe.fit(X_train, y_train)

    y_pred_train = pipe.predict(X_train)
    y_pred_test = pipe.predict(X_test)

    train_metrics = {
        'MAE': mean_absolute_error(y_train, y_pred_train),
        'MSE': mean_squared_error(y_train, y_pred_train),
        'RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'R2': r2_score(y_train, y_pred_train),
    }
    test_metrics = {
        'MAE': mean_absolute_error(y_test, y_pred_test),
        'MSE': mean_squared_error(y_test, y_pred_test),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'R2': r2_score(y_test, y_pred_test),
    }

    results[name] = {
        'pipeline': pipe,
        'train': train_metrics,
        'test': test_metrics,
    }

print('Entrenamiento completado.')

## 7. Evaluacion y Comparacion

In [ ]:
report_df = print_evaluation_report(results)

In [ ]:
for name, res in results.items():
    print(f'\n--- {name} ---')
    print(f'  Train R2: {res["train"]["R2"]:.4f}')
    print(f'  Test  R2: {res["test"]["R2"]:.4f}')
    gap = res['train']['R2'] - res['test']['R2']
    print(f'  Gap R2:   {gap:.4f}')
    if gap > 0.15:
        print(f'  ** Posible OVERFITTING **')
    elif res['test']['R2'] < 0:
        print(f'  ** Posible UNDERFITTING **')
    else:
        print(f'  OK')

## 8. Guardado del Modelo

In [ ]:
best_name = min(results, key=lambda k: results[k]['test']['RMSE'])
best_pipeline = results[best_name]['pipeline']

model_dir = '../models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'best_model.joblib')
joblib.dump(best_pipeline, model_path)

print(f'Mejor modelo: {best_name}')
print(f'Guardado en: {model_path}')

# Verificar que se puede cargar
loaded = joblib.load(model_path)
test_pred = loaded.predict(X_test.head(1))
print(f'Prediccion de prueba: {test_pred[0]:.2f}B')

## 9. Verificacion de No Data Leakage

El pipeline esta disenado para evitar data leakage:
- Feature engineering se aplica a todos los datos (no usa informacion del test)
- El preprocessor (imputacion, escalado, OHE) se fittea SOLO con train
- Test se transforma con los parametros aprendidos del train

In [ ]:
print('Verificacion de no data leakage:')
print(f'1. FeatureEngineer se fittea en train y transforma test (sin leakage)')
print(f'2. Preprocessor (imputer, scaler, encoder) se fittea solo en train')
print(f'3. Test se transforma con parametros del train')
print(f'4. Pipeline completo: FeatureEngineering -> Preprocessor -> Model')
print(f'\nEl modelo guardado contiene todo el pipeline y puede usarse directamente.')